In [41]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [42]:
df = pd.read_csv("../../output/03_aspi_handled.csv")

# Convert to correct types
df['date'] = pd.to_datetime(df['date'])

# Important numeric columns
df['close'] = pd.to_numeric(df['close'], errors='coerce')
df['aspi_close'] = pd.to_numeric(df['aspi_close'], errors='coerce')

# Sort properly
df = df.sort_values(['symbol', 'date'])

In [43]:
df.groupby('symbol')[['symbol', 'date']].head()

,symbol,date
4638,AAF,2025-02-03
4639,AAF,2025-02-05
4640,AAF,2025-02-06
4641,AAF,2025-02-07
4642,AAF,2025-02-10
...,...,...
48385,WAPO,2025-02-03
48386,WAPO,2025-02-05
48387,WAPO,2025-02-06
48388,WAPO,2025-02-07


In [44]:
# Stock return
df['R_i'] = df.groupby('symbol')['close'].pct_change()

# Market return (same for all stocks per day)
df['R_m'] = df.groupby('symbol')['aspi_close'].pct_change()

In [45]:
event_date = pd.to_datetime("2025-11-28")

df['day'] = (df['date'] - event_date).dt.days

# Estimation window
estimation_df = df[(df['day'] >= -120) & (df['day'] <= -6)]

In [46]:
results = []

for symbol, group in estimation_df.groupby('symbol'):

    # Drop NaNs
    group = group[['R_i', 'R_m']].dropna()

    if len(group) < 30:  # safety check
        continue

    X = group['R_m']
    y = group['R_i']

    # Add constant (for alpha)
    X = sm.add_constant(X)

    model = sm.OLS(y, X).fit()

    alpha = model.params['const']
    beta = model.params['R_m']

    results.append({
        'symbol': symbol,
        'alpha': alpha,
        'beta': beta
    })

alpha_beta_df = pd.DataFrame(results)

In [47]:
alpha_beta_df.head()

,symbol,alpha,beta
0,AAF,0.000897,0.158756
1,AAIC,-0.002666,0.654950
2,ABAN,0.008363,1.633533
3,ABL,0.002495,0.584640
4,ACAP,0.003096,1.401049


In [48]:
df = df.merge(alpha_beta_df, on='symbol', how='left')
df.head()

,symbol,date,open,high,low,close,volume,suffix,sector,suffix_cat,aspi_open,aspi_high,aspi_low,aspi_close,R_i,R_m,day,alpha,beta
0,AAF,2025-02-03,29.3,29.3,28.5,28.8,148508.0,N0000,Diversified Financials,0,17122.73,17127.65,16900.36,16956.49,NaN,NaN,-298,0.000897,0.158756
1,AAF,2025-02-05,28.8,28.9,27.8,27.9,155337.0,N0000,Diversified Financials,0,16956.49,16974.58,16440.74,16456.10,-0.031250,-0.029510,-296,0.000897,0.158756
2,AAF,2025-02-06,27.8,29.5,27.5,29.5,166488.0,N0000,Diversified Financials,0,16456.10,16675.80,16295.31,16506.73,0.057348,0.003077,-295,0.000897,0.158756
3,AAF,2025-02-07,29.0,29.1,28.5,28.6,228939.0,N0000,Diversified Financials,0,16506.73,16746.78,16506.38,16734.68,-0.030508,0.013810,-294,0.000897,0.158756
4,AAF,2025-02-10,28.6,29.7,28.6,29.0,14430.0,N0000,Diversified Financials,0,16734.68,16794.64,16537.98,16566.27,0.013986,-0.010064,-291,0.000897,0.158756


In [49]:
df.to_csv('../../output/04_Regression_handled.csv',index = False)